In [1]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [13]:
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
import pandas as pd

#==============================================================
# 1. Load Data
#==============================================================
df = pd.read_csv("ai_teacher_results.csv")


#===============================================================
# 2. Prepare Reference And Generated Answers
#===============================================================

references = (
    df["Teacher Response"].fillna("")
    .astype(str)
    .tolist()
)

candidates = (
    df["GeneratedAnswer"].fillna("")
    .astype(str)
    .tolist()
)

#=================================================================
# 3. BertScore
#=================================================================

P, R, F1 = bert_score(
    candidates,
    references,
    lang = "en",
    verbose = True
)

df["BERT"] = F1.cpu().numpy()

#=================================================================
# 4. Rouge:
#=================================================================

rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer = True)
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for candidate, reference in zip(candidates, references):
    scores = rouge.score(reference, candidate)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

df["ROUGE-1"] = rouge1_scores
df["ROUGE-2"] = rouge2_scores
df["ROUGE-L"] = rougeL_scores

#=================================================================
# 5. Meteor
#=================================================================

meteor_scores = []

for candidates, reference in zip(candidates, references):

    candidate_tokens = candidate.split()
    reference_tokens = reference.split()
    score = meteor_score([reference_tokens], candidate_tokens)
    meteor_scores.append(score)

df["METEOR"] = meteor_scores

#======================================================================
# 6. Round Score
#======================================================================

similarity_columns = [
    "BERT",
    "ROUGE-1",
    "ROUGE-2",
    "ROUGE-L",
    "METEOR"
]

df[similarity_columns] = (df[similarity_columns].round(3))

#=========================================================================
# 7. Question Wise Similarity DataFrame:
#=========================================================================

similarity_results = df[
    [
        "User Query",
        "BERT",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "METEOR"
    ]
    ].copy()


#  Display Question Wise DataFrame:

display(similarity_results)

#============================================================================
# 8. Calculate Average:
#============================================================================

average_bert = df["BERT"].mean()
average_rouge1 = df["ROUGE-1"].mean()
average_rouge2 = df["ROUGE-2"].mean()
average_rougeL = df["ROUGE-L"].mean()
average_meteor = df["METEOR"].mean()

#============================================================================
# 9. Create Average Similarity DataFrame:
#============================================================================

average_similarity = pd.DataFrame({
    "Metric": ["BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"],
    "Average Score": [average_bert, average_rouge1, average_rouge2, average_rougeL, average_meteor]
})

# Round Average Score:

average_similarity["Average Score"] = (average_similarity["Average Score"].round(3))

# Display Average DataFrame

display(average_similarity)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 18.80 seconds, 1.06 sentences/sec


,User Query,BERT,ROUGE-1,ROUGE-2,ROUGE-L,METEOR
0,What is AI?,0.930,0.571,0.393,0.571,0.056
1,How does AI differ from traditional programming?,0.920,0.635,0.393,0.571,0.115
2,What is Machine Learning?,0.919,0.557,0.390,0.405,0.082
3,What is Deep Learning?,0.905,0.551,0.328,0.406,0.057
4,"What is the difference between AI, ML, and Dee...",0.893,0.584,0.322,0.539,0.090
5,What is training data?,0.914,0.562,0.452,0.562,0.148
6,What is a Machine Learning model?,0.912,0.500,0.303,0.500,0.079
7,What is supervised learning?,0.876,0.286,0.080,0.208,0.019
8,What is unsupervised learning?,0.906,0.381,0.164,0.317,0.040
9,What is reinforcement learning?,0.922,0.517,0.286,0.414,0.069


,Metric,Average Score
0,BERT,0.902
1,ROUGE-1,0.465
2,ROUGE-2,0.257
3,ROUGE-L,0.396
4,METEOR,0.079


In [14]:
df.to_csv(
    "ai_teacher_evaluation_Similarity_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

In [15]:
average_similarity.to_csv("ai_teacher_evaluation_similarity_average.csv", index = False, encoding = "utf-8-sig")